# 08.6 — Running MATLAB analysis on Python output

This is the payoff of the whole interop module — and, in a sense, of the whole port. You've trained the model in Python (fast iteration, the PyTorch ecosystem), and it wrote results into the byte-exact MATLAB folder hierarchy ([08.1](08.1_folder_hierarchy_generation.ipynb)) as schema-perfect `CM_Table.mat` files ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) that pass the round-trip gate ([08.4](08.4_the_mat_round_trip_test.ipynb)). Now you run the *original, unmodified* MATLAB analysis scripts on that output — `DATA_cggAllNetworkEncoderResults.m` walks the tree, loads every table, and draws the same figures it always did. Train in Python, analyze in MATLAB. This notebook closes that loop.

This notebook covers:

* The end-to-end loop: Python trains, MATLAB analyzes, unmodified.
* How Python invokes MATLAB (`matlab -batch` subprocess, not the Engine).
* The `matlab_available()` gate for graceful MATLAB-optional code.
* Why byte-exact parity ([08.1](08.1_folder_hierarchy_generation.ipynb)–[08.4](08.4_the_mat_round_trip_test.ipynb)) is what makes this possible.

**Prerequisites:** [08.1](08.1_folder_hierarchy_generation.ipynb)–[08.4](08.4_the_mat_round_trip_test.ipynb) (the whole output chain).

## Section 1 — What MATLAB does

The MATLAB analysis pipeline is unchanged from the original project — it doesn't know or care that Python produced the results:

```matlab
% Walk the Encoding/ tree, load every Fold_N/CM_Table.mat, aggregate:
results = DATA_cggAllNetworkEncoderResults(baseDir);   % discovers runs by folder (08.1)
FIGURE_cggAllNetworkEncoderResults(results);            % draws confusion matrices, accuracy bars
```

These scripts glob the folder hierarchy, `load` each `CM_Table.mat`, read `Aggregation_Prediction` vs `TrueValue` to compute accuracy, and plot. Because the Python output *is* byte-compatible ([08.4](08.4_the_mat_round_trip_test.ipynb)), the analysis runs verbatim. The only new piece is a way for Python to *invoke* MATLAB when you want to automate that step — `matlab_runner`.

## Section 2 — The Python concepts you need

### 2.1 — The payoff: split the pipeline at the results file

The entire interop effort buys one thing: you can **replace the training half in Python while keeping the analysis half in MATLAB**. The `CM_Table.mat` file is the seam. Upstream of it, Python owns the work (data loading, the VAE, the curriculum, training). Downstream, MATLAB owns it (aggregation, statistics, figures for the paper). Neither half needs to know about the other's language — they meet at a `.mat` file in a well-known folder. This is why parity was worth the effort: not to rewrite MATLAB, but to *interoperate* with it, so years of analysis tooling keep working.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3.6))
def box(x, label, color, w=2.6):
    ax.add_patch(mpatches.FancyBboxPatch((x - w/2, 0.7), w, 1.0,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", lw=1.3))
    ax.text(x, 1.2, label, ha="center", va="center", fontsize=9.5, fontweight="bold")
def arrow(x1, x2, label):
    ax.annotate("", xy=(x2, 1.2), xytext=(x1, 1.2), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.text((x1 + x2)/2, 1.75, label, ha="center", fontsize=8.5, color="#333")

box(1.6, "PYTHON\ntrain the model", "#cce4ff")
box(6.0, "CM_Table.mat\n(the seam)", "#ffe4cc", w=2.4)
box(10.4, "MATLAB\naggregate + plot", "#e6f0d0")
arrow(2.9, 4.8, "writes (08.1, 08.2)")
arrow(7.2, 9.1, "loads unmodified")
ax.text(6.0, 0.35, "byte-exact parity (08.4) is what lets the seam hold", ha="center", fontsize=8.5, style="italic", color="#555")
ax.set_xlim(0, 12); ax.set_ylim(0, 2.4); ax.axis("off")
ax.set_title("Train in Python → results at the seam → analyze in MATLAB")
plt.tight_layout(); plt.show()
print("The .mat file is the language-agnostic contract between the two halves.")

### 2.2 — How Python invokes MATLAB

When you want to *automate* the analysis step (run MATLAB from a Python script), `matlab_runner` drives MATLAB in **batch mode** — `matlab -batch "..."` as a subprocess — rather than the MATLAB Engine for Python:

In [ ]:
from neural_data_decoding.interop.matlab_runner import (
    matlab_available, quote_matlab_path, run_matlab_batch)

# quote_matlab_path is pure-Python (no MATLAB needed) — safe path quoting:
print("quoted path:", quote_matlab_path("/Neural Data Reading/CM_Table.mat"))
print()
# matlab_available() gates any actual invocation:
print("MATLAB available on this machine?", matlab_available())

Why a subprocess and not the MATLAB Engine? **Architecture independence.** The MATLAB Engine for Python binds the Python interpreter to a MATLAB of the *same* CPU architecture — which breaks on Apple Silicon when a Rosetta-translated (x86-64) Python is paired with a native (ARM64) MATLAB. The `matlab -batch` subprocess sidesteps this entirely: MATLAB runs in its own process at its native architecture, and Python just shells out to it. (`matlab_runner` even detects Rosetta via `sysctl` and prefixes `arch -arm64` so the launcher finds the right binary.) It's the robust, portable way to call MATLAB from any Python.

### 2.3 — The `matlab_available()` gate

In [ ]:
# Guard any MATLAB invocation so the code runs cleanly WITHOUT MATLAB too,
# AND wrap the spawn so a flaky/unlicensed MATLAB can't crash the pipeline.
if matlab_available():
    try:
        result = run_matlab_batch("disp(2 + 2)", capture_output=True, timeout=120)
        print("MATLAB ran; 2 + 2 =", result.stdout.strip().splitlines()[-1])
        print("→ the subprocess bridge works: Python → matlab -batch → output back to Python.")
    except Exception as e:
        print(f"MATLAB present but the batch call failed ({type(e).__name__}) — analysis step skipped.")
        print("→ wrapping the spawn keeps a flaky/unlicensed MATLAB from crashing the pipeline.")
else:
    print("MATLAB not found — skipping the live invocation.")
    print("→ matlab_available() lets MATLAB-dependent code degrade gracefully (returns False).")

`matlab_available()` returns `True`/`False` so code can *choose* whether to run the MATLAB step. This is the same pattern as the `needs_matlab` pytest marker ([08.4 §2.5](08.4_the_mat_round_trip_test.ipynb)): the MATLAB-dependent path is optional. CI (no MATLAB) skips it; a workstation with MATLAB runs it. Your automation script guards the analysis call with this check, so it works everywhere — running the full loop where MATLAB exists, and stopping cleanly at the `.mat` files where it doesn't.

### 2.4 — The aggregator discovers runs by folder

The reason `DATA_cggAllNetworkEncoderResults.m` needs *no* modification is the folder hierarchy ([08.1](08.1_folder_hierarchy_generation.ipynb)). It doesn't take a list of runs — it *discovers* them by globbing the `Encoding/` tree for `Fold_N/CM_Table.mat` files. Because Python wrote its results into the exact same tree with the exact same filenames, the aggregator finds Python runs identically to MATLAB runs. It loads each table, reads `Aggregation_Prediction` vs `TrueValue`, tallies accuracy per config, and groups folds — all keyed off the path structure. The byte-exact naming ([08.1 §2.5](08.1_folder_hierarchy_generation.ipynb)) and the schema-perfect table ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) are precisely what make the discovery + load succeed unmodified.

### 2.5 — The complete Module 08 picture

Putting the whole module together, the output-and-analysis flow is:

1. Training finishes; the run resolves its leaf path in the folder hierarchy ([08.1](08.1_folder_hierarchy_generation.ipynb)).
2. On each new-best epoch, `CM_Table_Validation.mat` and (at the end) `CM_Table.mat` are written there ([08.2](08.2_writing_mat_files_for_matlab.ipynb), [08.3](08.3_monitor_table_compatibility.ipynb)).
3. The round-trip gate ([08.4](08.4_the_mat_round_trip_test.ipynb)) guarantees those files are MATLAB-readable; W&B ([08.5](08.5_weights_and_biases_integration.ipynb)) optionally streams live telemetry alongside.
4. The MATLAB aggregator (this notebook) walks the tree, loads the tables, and produces the final figures — unchanged from the original pipeline.

Every notebook in the module is one link in that chain, and byte-exact MATLAB parity is the thread running through all of them.

## Section 3 — The `neural_data_decoding` implementation

`run_matlab_batch` — the subprocess bridge:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/interop/matlab_runner.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def run_matlab_batch"))
for k in range(i, i + 9):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `run_matlab_batch(commands, *, cwd=None, check=True, capture_output=False, timeout=None)` — runs `matlab -batch "<commands>"` as a subprocess and returns a `subprocess.CompletedProcess`. `check=True` raises on a non-zero MATLAB exit; `capture_output=True` returns stdout/stderr.
* `find_matlab_executable()` searches `$MATLAB_EXECUTABLE` → `$PATH` → the standard install locations (`/Applications/MATLAB_R*.app/bin/matlab`, newest wins); `matlab_available()` wraps it in a `try` to return a bool ([§2.3](08.6_running_matlab_analysis_on_python_output.ipynb)).
* `quote_matlab_path` single-quotes paths and doubles internal quotes — needed because the MATLAB source lives under a space-containing path (`Neural Data Reading`), which would otherwise break the `-batch` string.
* The Apple-Silicon `arch -arm64` prefix (module docstring) is applied automatically when a Rosetta Python is detected — the cross-architecture robustness (§2.2).
* This runner is shared by the `needs_matlab` interop tests ([08.4](08.4_the_mat_round_trip_test.ipynb)) and the golden-fixture generation script — one implementation for every Python→MATLAB call.

## Section 4 — Hands-on exercises

### Exercise 4.1 — a MATLAB-optional analysis function

Write a function that runs a MATLAB analysis *if* MATLAB is available and otherwise returns a clear "skipped" result — the graceful-degradation pattern.

In [ ]:
# Reveal:
def maybe_run_analysis(command):
    if not matlab_available():
        return {"ran": False, "reason": "MATLAB not found"}
    try:
        result = run_matlab_batch(command, capture_output=True, timeout=120)
        return {"ran": True, "output": result.stdout.strip().splitlines()[-1]}
    except Exception as e:
        return {"ran": False, "reason": f"invocation failed: {type(e).__name__}"}

decision = "would run" if matlab_available() else "would skip (no MATLAB)"
print(f"analysis command 'disp(6*7)' → the guard decides: {decision}")
print("→ maybe_run_analysis(cmd) runs it where MATLAB exists, returns a 'skipped' dict where it doesn't.")
print("  Same code path everywhere — the availability check, not a crash, decides.")

### Exercise 4.2 — the seam is language-agnostic

Confirm that the `.mat` a Python run writes ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) is exactly what the MATLAB aggregator expects to load — by checking a written file has the aggregator's required field.

In [ ]:
# Reveal:
import numpy as np, scipy.io, tempfile, os
from neural_data_decoding.interop.cm_table_format import write_cm_table_mat
p = os.path.join(tempfile.mkdtemp(), "CM_Table.mat")
write_cm_table_mat(p, data_numbers=np.arange(3) + 1, true_values=np.zeros((3, 2)),
                   window_predictions=[np.zeros((3, 2))], aggregation_prediction=None)
fields = set(scipy.io.loadmat(p)["CM_Table"].dtype.names)
print("aggregator needs 'Aggregation_Prediction' and 'TrueValue' — present?",
      {"Aggregation_Prediction", "TrueValue"} <= fields)
print("→ the Python-written file carries exactly the fields MATLAB's analysis reads.")

## Section 5 — Common errors

### `MatlabNotFoundError` / MATLAB not found

`find_matlab_executable` couldn't locate MATLAB. Set `$MATLAB_EXECUTABLE`, add MATLAB to `$PATH`, or install it. For code that should tolerate its absence, guard with `matlab_available()` (§2.3) instead of calling blindly.

### The aggregator finds no Python runs

A folder-hierarchy mismatch ([08.1 §2.5](08.1_folder_hierarchy_generation.ipynb)) — the Python output isn't under the exact tree/names the aggregator globs. Or the analysis's `baseDir` points somewhere else. Check the path the run actually wrote to.

### MATLAB errors loading the `.mat`

A schema/format problem ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) — run the round-trip gate ([08.4](08.4_the_mat_round_trip_test.ipynb)) to localize it. If the pure-Python round-trip passes but MATLAB still errors, run the `needs_matlab` level for the real MATLAB error.

### Wrong architecture on Apple Silicon

The MATLAB Engine won't bind a Rosetta Python to an ARM64 MATLAB (§2.2) — that's *why* this project uses the `matlab -batch` subprocess. Don't reach for `matlabengine`; use `run_matlab_batch`.

### Path with spaces breaks the `-batch` string

Use `quote_matlab_path` (§3) for any path in a MATLAB command — the source root's `Neural Data Reading` folder has a space that unquoted would split the command.

## Section 6 — Further reading

- [`src/neural_data_decoding/interop/matlab_runner.py`](../../src/neural_data_decoding/interop/matlab_runner.py) — the subprocess bridge.
- [`src/neural_data_decoding/interop/matlab_table_writer.py`](../../src/neural_data_decoding/interop/matlab_table_writer.py) — promoting a struct to a native MATLAB `table` for analysis scripts that need `istable`.
- [MATLAB `-batch` docs](https://www.mathworks.com/help/matlab/ref/matlablinux.html) — the non-interactive mode this uses.

**Module 08 complete.** You've traced the full output-and-analysis path: the config-encoded folder hierarchy, the `CM_Table.mat` schema, the monitor/model-selection callbacks, the round-trip parity gate, W&B telemetry, and finally running the unmodified MATLAB analysis on Python output. Next up: **[Module 09 — production deployment](../09_production_deployment/)**, where the pipeline scales onto a compute cluster with SLURM.